# Exercises XP - Heart Disease Prediction

## What you will learn
- Load and inspect CSV data
- EDA and preprocessing
- Train Logistic Regression, SVM, XGBoost
- Hyperparameter tuning with GridSearchCV
- Evaluate with standard metrics

## What you will create
- Working classifiers and a simple comparison report


## Setup

In [ ]:
import os, zipfile, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

try:
    from xgboost import XGBClassifier
except ImportError:
    # In Colab or local env, uncomment below to install if needed
    # !pip install xgboost
    XGBClassifier = None

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## Exercise 1 - Exploratory Data Analysis

In [ ]:
# Load the dataset
csv_path = 'dataset_heart.csv'
df = pd.read_csv(csv_path)

# Basic Inspection
print("Shape:", df.shape)
print(df.info())
print(df.head())

# Check for nulls
print("\nMissing Values:\n", df.isnull().sum())

# Identify target column and map to 0/1
# The dataset uses 1 (No Disease) and 2 (Disease)
target = 'heart disease'
df[target] = df[target].map({1: 0, 2: 1})

# Split features and target
X = df.drop(columns=[target])
y = df[target]

# Train test split with stratification
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
print(f"\nTrain size: {X_train.shape}, Test size: {X_test.shape}")

### Basic visual checks

In [ ]:
# Plot histograms for numeric columns
num_cols = ['age', 'resting blood pressure', 'serum cholestoral', 'max heart rate', 'oldpeak']
df[num_cols].hist(figsize=(12, 8), bins=20)
plt.tight_layout()
plt.show()

# Plot class balance
sns.countplot(x=target, data=df, palette='magma')
plt.title('Class Balance (0: No Disease, 1: Disease)')
plt.show()

## Preprocessing pipeline

In [ ]:
# Define categorical and numeric columns
cat_cols = ['sex ', 'chest pain type', 'fasting blood sugar', 'resting electrocardiographic results', 'exercise induced angina', 'ST segment', 'major vessels', 'thal']
num_cols = ['age', 'resting blood pressure', 'serum cholestoral', 'max heart rate', 'oldpeak']

# Build ColumnTransformer
pre = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

## Helper - evaluation function

In [ ]:
def eval_and_report(name, model, X_test, y_test):
    """Compute metrics and draw confusion matrix and ROC if available."""
    y_pred = model.predict(X_test)
    
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
    }
    
    print(f"--- {name} Results ---")
    for k, v in metrics.items():
        print(f"{k.capitalize()}: {v:.4f}")

    # Confusion Matrix
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm).plot(ax=ax[0])
    ax[0].set_title('Confusion Matrix')

    # ROC Curve
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        ax[1].plot(fpr, tpr, label=f"AUC: {roc_auc_score(y_test, y_prob):.4f}")
        ax[1].plot([0, 1], [0, 1], 'k--')
        ax[1].set_xlabel('False Positive Rate')
        ax[1].set_ylabel('True Positive Rate')
        ax[1].set_title('ROC Curve')
        ax[1].legend()
    
    plt.show()
    return metrics

## Exercise 2 - Logistic Regression without Grid Search

In [ ]:
# Pipeline with LogisticRegression
pipe_lr = Pipeline([
    ('pre', pre),
    ('lr', LogisticRegression(solver='liblinear', random_state=RANDOM_STATE))
])

# Fit on training data
pipe_lr.fit(X_train, y_train)

# Evaluate
lr_no_gs_metrics = eval_and_report('Logistic Regression (No Grid Search)', pipe_lr, X_test, y_test)

## Exercise 3 - Logistic Regression with Grid Search

In [ ]:
pipe_lr_cv = Pipeline([
    ('pre', pre),
    ('lr', LogisticRegression(solver='liblinear', random_state=RANDOM_STATE))
])

param_grid = {
    'lr__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'lr__penalty': ['l1', 'l2']
}

grid_lr = GridSearchCV(pipe_lr_cv, param_grid, cv=5, scoring='f1', n_jobs=-1)
grid_lr.fit(X_train, y_train)

print('Best params:', grid_lr.best_params_)
best_lr = grid_lr.best_estimator_
lr_gs_metrics = eval_and_report('Logistic Regression (Grid Search)', best_lr, X_test, y_test)

## Exercise 4 - SVM without Grid Search

In [ ]:
pipe_svm = Pipeline([
    ('pre', pre),
    ('svm', SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=RANDOM_STATE))
])

pipe_svm.fit(X_train, y_train)
svm_no_gs_metrics = eval_and_report('SVM (No Grid Search)', pipe_svm, X_test, y_test)

## Exercise 5 - SVM with Grid Search

In [ ]:
pipe_svm_cv = Pipeline([
    ('pre', pre),
    ('svm', SVC(probability=True, random_state=RANDOM_STATE))
])

svm_param_grid = {
    'svm__kernel': ['rbf', 'linear', 'poly'],
    'svm__C': [0.1, 1, 10, 100],
    'svm__gamma': ['scale', 'auto']
}

grid_svm = GridSearchCV(pipe_svm_cv, svm_param_grid, cv=5, scoring='f1', n_jobs=-1)
grid_svm.fit(X_train, y_train)

print('Best params:', grid_svm.best_params_)
best_svm = grid_svm.best_estimator_
svm_gs_metrics = eval_and_report('SVM (Grid Search)', best_svm, X_test, y_test)

## Exercise 6 - XGBoost without Grid Search

In [ ]:
if XGBClassifier:
    # Hyperparameters manually chosen to avoid overfitting on small dataset
    pipe_xgb = Pipeline([
        ('pre', pre),
        ('xgb', XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE))
    ])
    pipe_xgb.fit(X_train, y_train)
    xgb_no_gs_metrics = eval_and_report('XGBoost (No Grid Search)', pipe_xgb, X_test, y_test)
else:
    print("XGBoost not installed.")
    xgb_no_gs_metrics = None

## Exercise 7 - XGBoost with Grid Search

In [ ]:
if XGBClassifier:
    pipe_xgb_cv = Pipeline([
        ('pre', pre),
        ('xgb', XGBClassifier(random_state=RANDOM_STATE))
    ])

    xgb_param_grid = {
        'xgb__n_estimators': [50, 100, 200],
        'xgb__learning_rate': [0.01, 0.1, 0.2],
        'xgb__max_depth': [3, 4, 5],
        'xgb__subsample': [0.8, 1.0]
    }

    grid_xgb = GridSearchCV(pipe_xgb_cv, xgb_param_grid, cv=5, scoring='f1', n_jobs=-1)
    grid_xgb.fit(X_train, y_train)

    print('Best params:', grid_xgb.best_params_)
    best_xgb = grid_xgb.best_estimator_
    xgb_gs_metrics = eval_and_report('XGBoost (Grid Search)', best_xgb, X_test, y_test)
else:
    print("XGBoost not installed.")
    xgb_gs_metrics = None

## Compare models

In [ ]:
summary = {}
summary['LR no grid'] = lr_no_gs_metrics
summary['LR grid'] = lr_gs_metrics
summary['SVM no grid'] = svm_no_gs_metrics
summary['SVM grid'] = svm_gs_metrics
if xgb_no_gs_metrics:
    summary['XGB no grid'] = xgb_no_gs_metrics
if xgb_gs_metrics:
    summary['XGB grid'] = xgb_gs_metrics

comparison_df = pd.DataFrame.from_dict(summary, orient='index')
print("Model Performance Comparison:")
print(comparison_df.sort_values(by='f1', ascending=False))

# Visualization of F1 Score
comparison_df['f1'].plot(kind='bar', figsize=(10, 6), color='skyblue')
plt.title('F1-Score Comparison Across Models')
plt.ylabel('F1 Score')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()